**Feb Internship NLP - Build a Chatbot using Hugging Face Transformers.**         

In [5]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# NLP – 3 Assignment: Chatbot using Transformers\n",
    "\n",
    "**Student Name:** Your Name\n",
    "\n",
    "**Objective:** Build a conversational chatbot using Hugging Face Transformers that can interact with users and generate meaningful responses.\n",
    "\n",
    "**Tools Used:** Python, PyTorch/TensorFlow, Hugging Face Transformers, Jupyter/Colab\n",
    "\n",
    "### Learning Outcomes\n",
    "1. Understand how transformer-based language models work\n",
    "2. Use pre-trained NLP models from Hugging Face Model Hub\n",
    "3. Perform text generation using transformer models\n",
    "4. Build an interactive conversational system\n",
    "5. Understand prompt-based text generation\n",
    "\n",
    "### Sample Conversation\n",
    "Chatbot: Hello! I am your AI assistant. Type 'exit' or 'quit' to stop.\n",
    "User: Hello\n",
    "Chatbot: Hi there! How can I help you today?\n",
    "User: Who created Python?\n",
    "Chatbot: Python was created by Guido van Rossum and released in 1991.\n",
    "User: What is AI?\n",
    "Chatbot: Artificial Intelligence refers to machines simulating human intelligence.\n",
    "User: exit\n",
    "Chatbot: Goodbye! Have a nice day."
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "source": [
    "!pip install transformers --quiet\n",
    "\n",
    "from transformers import AutoModelForCausalLM, AutoTokenizer\n",
    "import torch\n",
    "\n",
    "# Load DialoGPT-small model\n",
    "tokenizer = AutoTokenizer.from_pretrained(\"microsoft/DialoGPT-small\")\n",
    "model = AutoModelForCausalLM.from_pretrained(\"microsoft/DialoGPT-small\")\n",
    "\n",
    "print(\"Chatbot: Hello! I am your AI assistant. Type 'exit' or 'quit' to stop.\")\n",
    "\n",
    "# Chat loop\n",
    "while True:\n",
    "    user_input = input(\"User: \")\n",
    "    if user_input.lower() in [\"exit\", \"quit\"]:\n",
    "        print(\"Chatbot: Goodbye! Have a nice day.\")\n",
    "        break\n",
    "\n",
    "    # Encode user input + previous context\n",
    "    input_ids = tokenizer.encode(user_input + tokenizer.eos_token, return_tensors=\"pt\")\n",
    "\n",
    "    # Generate response\n",
    "    chat_history_ids = model.generate(input_ids, max_length=1000, pad_token_id=tokenizer.eos_token_id)\n",
    "    bot_response = tokenizer.decode(chat_history_ids[:, input_ids.shape[-1]:][0], skip_special_tokens=True)\n",
    "\n",
    "    print(f\"Chatbot: {bot_response}\")"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "name": "python3",
   "display_name": "Python 3"
  },
  "language_info": {
   "name": "python",
   "version": "3.11"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}

{'cells': [{'cell_type': 'markdown',
   'metadata': {},
   'source': ['# NLP – 3 Assignment: Chatbot using Transformers\n',
    '\n',
    '**Student Name:** Your Name\n',
    '\n',
    '**Objective:** Build a conversational chatbot using Hugging Face Transformers that can interact with users and generate meaningful responses.\n',
    '\n',
    '**Tools Used:** Python, PyTorch/TensorFlow, Hugging Face Transformers, Jupyter/Colab\n',
    '\n',
    '### Learning Outcomes\n',
    '1. Understand how transformer-based language models work\n',
    '2. Use pre-trained NLP models from Hugging Face Model Hub\n',
    '3. Perform text generation using transformer models\n',
    '4. Build an interactive conversational system\n',
    '5. Understand prompt-based text generation\n',
    '\n',
    '### Sample Conversation\n',
    "Chatbot: Hello! I am your AI assistant. Type 'exit' or 'quit' to stop.\n",
    'User: Hello\n',
    'Chatbot: Hi there! How can I help you today?\n',
    'User: Who created P

In [ ]:
# Step 1: Install libraries
!pip install transformers
!pip install torch

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Step 2: Load DialoGPT model
tokenizer = AutoTokenizer.from_pretrained("microsoft/DialoGPT-medium")
model = AutoModelForCausalLM.from_pretrained("microsoft/DialoGPT-medium")

# Step 3: Simple factual knowledge base
fact_knowledge = {
    "who created python": "Python was created by Guido van Rossum and released in 1991.",
    "what is python": "Python is a high-level, interpreted programming language created by Guido van Rossum.",
    "what is artificial intelligence": "Artificial Intelligence refers to the simulation of human intelligence by machines that can perform tasks such as learning, reasoning, and problem solving.",
    "who is guido van rossum": "Guido van Rossum is the creator of Python programming language.",
    "what is machine learning": "Machine Learning is a subset of AI that enables systems to learn from data and improve from experience without being explicitly programmed."
}

# Step 4: Initialize chat history
chat_history_ids = None

# Step 5: Start chatbot
print("Chatbot: Hello! I am your AI assistant. Type 'exit' or 'quit' to stop.")

while True:
    user_input = input("User: ").strip()

    # Step 5.1: Exit condition
    if user_input.lower() in ["exit", "quit"]:
        print("Chatbot: Goodbye! Have a nice day.")
        break

    # Step 5.2: Check factual knowledge first
    lower_input = user_input.lower()
    response = None
    for key in fact_knowledge:
        if key in lower_input:
            response = fact_knowledge[key]
            break

    # Step 5.3: If not a factual question, use DialoGPT
    if response is None:
        prompt_text = user_input  # keep normal prompt
        new_input_ids = tokenizer.encode(prompt_text + tokenizer.eos_token, return_tensors='pt')

        # Append previous chat history
        if chat_history_ids is not None:
            bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1)
        else:
            bot_input_ids = new_input_ids

        # Generate model response
        chat_history_ids = model.generate(
            bot_input_ids,
            attention_mask=torch.ones_like(bot_input_ids),  # fix attention warning
            max_length=1000,
            pad_token_id=tokenizer.eos_token_id,
            no_repeat_ngram_size=3,
            do_sample=True,
            top_k=50,
            top_p=0.9,
            temperature=0.7
        )

        response = tokenizer.decode(chat_history_ids[:, bot_input_ids.shape[-1]:][0], skip_special_tokens=True)

    # Step 5.4: Print response
    print(f"Chatbot: {response}")

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-medium
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Chatbot: Hello! I am your AI assistant. Type 'exit' or 'quit' to stop.
User: Who created python?
Chatbot: Python was created by Guido van Rossum and released in 1991.
User: What is Artificial Intelligence?
Chatbot: Artificial Intelligence refers to the simulation of human intelligence by machines that can perform tasks such as learning, reasoning, and problem solving.
User: What is machine learning?
Chatbot: Machine Learning is a subset of AI that enables systems to learn from data and improve from experience without being explicitly programmed.
User: How are you?
Chatbot: Fine. You?
User: exit
Chatbot: Goodbye! Have a nice day.
